In [1]:
"""
Week 7 — Advanced Models (Gradient Boosting) 

Deliverable: test-set R2 for a gradient-boosted regressor (XGBoost), after a
LIGHT hyperparameter search — but one informed by a prior wide search rather
than a blind guess.

--- Why this grid, specifically -------------------------------------------
An earlier wide search (max_depth in [3,5,7,9,11], learning_rate in
[0.01,0.03,0.05,0.1,0.2,0.3], n_estimators found via early stopping)
produced two clear findings this grid is built around:

1. For learning_rate >= 0.05, validation R2 has a genuine INTERIOR peak
   around max_depth 7-9, then turns over (e.g. lr=0.05 peaked at depth=9,
   R2=0.8678, then dropped at depth=11). That is a real bias/variance
   tradeoff, not a search artifact.

2. The nominal wide-search "winner" (max_depth=11, learning_rate=0.01,
   ~1956 trees) is NOT a good pick despite the highest validation R2
   (0.8697): it beat the depth=9/lr=0.05 result (0.8678) by only 0.0019 —
   noise-level on a single validation month — while roughly DOUBLING the
   train-test R2 gap (0.089 vs. an expected ~0.04-0.05 in that region) and
   shifting feature importance toward missing-value indicators
   (PoolPrivateYN_Unknown, LivingArea_missing) rather than intrinsic
   property features. That is the signature of overfitting to the
   validation month's idiosyncrasies, not a genuinely better model.

This script therefore searches a small, DELIBERATELY interior grid centered
on the region that already showed a real peak, instead of re-running (or
further widening) the full search:
    max_depth     in [7, 9]
    learning_rate in [0.03, 0.05, 0.1]
(6 combinations — "light" by count, but not blind: every value here is
inside a region already shown to behave well, not a boundary being probed
for the first time.)

n_estimators is still found via early stopping rather than grid-searched,
for the same reason as before: it lets each learning_rate use exactly as
many trees as it needs.
-----------------------------------------------------------------------------

Other design choices carried over unchanged from prior weeks:

1. Feature set: cleaned_single_family_sales_engineered.csv (Week 6 output —
   BedBathRatio, PropertyAge, Unified school-district target encoding).

2. Leakage-safe machinery unchanged from Weeks 4-6: outlier thresholds fit
   on the training split only (Sec. 03), CV-safe target encoding for City
   and DistrictName (Sec. 05), Pipeline/ColumnTransformer fit exclusively on
   the training window (Sec. 07).

3. Training window X is fixed at 24 months (as in every prior week's best
   result) rather than jointly tuned with hyperparameters — still flagged
   as a next step in Sec. 10 below.

4. Early stopping's eval_set is the validation month only (model
   selection, Sec. 01) — never the test month. The final model is refit
   from scratch on the window before the test month with a plain, fixed
   n_estimators, then scored once on the test month.

5. Only R2 is reported for test metrics, per this week's spec.
"""

'\nWeek 7 — Advanced Models (Gradient Boosting) \n\nDeliverable: test-set R2 for a gradient-boosted regressor (XGBoost), after a\nLIGHT hyperparameter search — but one informed by a prior wide search rather\nthan a blind guess.\n\n--- Why this grid, specifically -------------------------------------------\nAn earlier wide search (max_depth in [3,5,7,9,11], learning_rate in\n[0.01,0.03,0.05,0.1,0.2,0.3], n_estimators found via early stopping)\nproduced two clear findings this grid is built around:\n\n1. For learning_rate >= 0.05, validation R2 has a genuine INTERIOR peak\n   around max_depth 7-9, then turns over (e.g. lr=0.05 peaked at depth=9,\n   R2=0.8678, then dropped at depth=11). That is a real bias/variance\n   tradeoff, not a search artifact.\n\n2. The nominal wide-search "winner" (max_depth=11, learning_rate=0.01,\n   ~1956 trees) is NOT a good pick despite the highest validation R2\n   (0.8697): it beat the depth=9/lr=0.05 result (0.8678) by only 0.0019 —\n   noise-level on a 

In [2]:
import numpy as np
import pandas as pd
import warnings
from itertools import product

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
# Gradient boosting library. XGBoost is used here; swapping in LightGBM only
# requires changing make_regressor() below (lightgbm.LGBMRegressor takes the
# same hyperparameter names).
from xgboost import XGBRegressor

In [5]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [6]:
DATA_PATH = "cleaned_single_family_sales_engineered.csv"   # Week 6 output

In [7]:
TARGET = "ClosePrice"
LOW_CARD_CAT = [
    "PropertySubType", "Levels", "AttachedGarageYN",
    "PoolPrivateYN", "FireplaceYN", "NewConstructionYN",
]
HIGH_CARD_CAT = ["City", "DistrictName"]

In [8]:
BASE_NUMERIC_BASE_NAMES = [
    "LivingArea", "Bedrooms", "Bathrooms", "LotSize",
    "GarageSpaces", "YearBuilt", "ParkingTotal",
]
ENGINEERED_BASE_NAMES = ["BedBathRatio", "PropertyAge"]

In [9]:
# Fixed training-window length (months) — see Design choice 3 above.
FIXED_X = 24

In [10]:
# ---------------------------------------------------------------------------
# Load enriched data
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["CloseDate"])
print(f"Loaded {DATA_PATH} — shape {df.shape}")

Loaded cleaned_single_family_sales_engineered.csv — shape (333028, 29)


In [11]:
# Defensive re-normalization (same rationale as every prior week: guards
# against bool/str mixed-type columns if this CSV is ever regenerated).
for col in HIGH_CARD_CAT + LOW_CARD_CAT:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [12]:
NUMERIC_COLS = BASE_NUMERIC_BASE_NAMES + ENGINEERED_BASE_NAMES
NUMERIC_COLS += [f"{c}_missing" for c in NUMERIC_COLS if f"{c}_missing" in df.columns]
print("Numeric columns:", NUMERIC_COLS)

Numeric columns: ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize', 'GarageSpaces', 'YearBuilt', 'ParkingTotal', 'BedBathRatio', 'PropertyAge', 'LivingArea_missing', 'Bathrooms_missing', 'LotSize_missing', 'GarageSpaces_missing', 'YearBuilt_missing', 'ParkingTotal_missing', 'BedBathRatio_missing', 'PropertyAge_missing']


In [13]:
# ---------------------------------------------------------------------------
# Chronological split helpers (Sec. 01) — unchanged from prior weeks
# ---------------------------------------------------------------------------
def month_period(series):
    return series.dt.to_period("M")

In [14]:
def get_window(data, end_month_exclusive, n_months):
    start = end_month_exclusive - n_months
    periods = month_period(data["CloseDate"])
    return data[(periods >= start) & (periods < end_month_exclusive)]

In [15]:
def get_month(data, month):
    return data[month_period(data["CloseDate"]) == month]

In [16]:
# ---------------------------------------------------------------------------
# CV-safe target encoding (Sec. 05) — unchanged from prior weeks
# ---------------------------------------------------------------------------
def kfold_target_encode(train_series, train_target, apply_series_list,
                         n_splits=5, smoothing=20, seed=RANDOM_SEED):
    train_series = train_series.reset_index(drop=True)
    train_target = train_target.reset_index(drop=True)
    global_mean = train_target.mean()

    encoded_train = np.zeros(len(train_series))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, hold_idx in kf.split(train_series):
        fit_vals, fit_target = train_series.iloc[fit_idx], train_target.iloc[fit_idx]
        stats = fit_target.groupby(fit_vals).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        encoded_train[hold_idx] = train_series.iloc[hold_idx].map(smoothed).fillna(global_mean).to_numpy()

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    encoded_apply = [s.map(full_smoothed).fillna(global_mean).to_numpy() for s in apply_series_list]
    return encoded_train, encoded_apply

In [17]:
def compute_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred)}

In [18]:
def build_preprocessor(numeric_final_cols, cat_final_cols):
    """Leak-safe ColumnTransformer (Sec. 07), factored out on its own so the
    early-stopping search can fit it once and reuse the transformed arrays,
    instead of going through the full Pipeline (which has no clean way to
    hand XGBoost a *preprocessed* eval_set for early stopping)."""
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_final_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", min_frequency=10)),
        ]), cat_final_cols),
    ])

In [19]:
def build_pipeline(regressor, numeric_final_cols, cat_final_cols):
    """Leak-safe full Pipeline (preprocess + regressor). Only the final
    regressor step changes between models."""
    preprocessor = build_preprocessor(numeric_final_cols, cat_final_cols)
    return Pipeline([("preprocess", preprocessor), ("regressor", regressor)])

In [20]:
def prepare_train_eval(train_df, eval_df, numeric_cols, high_card_cat):
    """Shared prep step: outlier filtering (Sec. 03, train-only thresholds)
    and CV-safe target encoding (Sec. 05)."""
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    lo, hi = train_df[TARGET].quantile([0.005, 0.995])
    train_df = train_df[(train_df[TARGET] >= lo) & (train_df[TARGET] <= hi)]
    eval_df = eval_df[(eval_df[TARGET] >= lo) & (eval_df[TARGET] <= hi)]

    y_train = train_df[TARGET].reset_index(drop=True)
    y_eval = eval_df[TARGET].reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)
    X_eval = eval_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)

    enc_cols = []
    for col in high_card_cat:
        enc_train, (enc_eval,) = kfold_target_encode(X_train[col], y_train, [X_eval[col]])
        enc_name = f"{col}_target_enc"
        X_train[enc_name] = enc_train
        X_eval[enc_name] = enc_eval
        enc_cols.append(enc_name)

    numeric_final = numeric_cols + enc_cols
    return X_train, y_train, X_eval, y_eval, numeric_final

In [21]:
def fit_and_evaluate(train_df, eval_df, regressor, numeric_cols, low_card_cat, high_card_cat):
    """Fits the full leak-safe pipeline on train_df only, scores on both
    train_df and eval_df. Train R2 is an overfitting diagnostic, not a
    formal metric. Used for the FINAL fit (plain, fixed n_estimators — no
    early stopping, no test-month involvement in choosing hyperparameters)."""
    X_train, y_train, X_eval, y_eval, numeric_final = prepare_train_eval(
        train_df, eval_df, numeric_cols, high_card_cat
    )
    model = build_pipeline(regressor, numeric_final, low_card_cat)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
        model.fit(X_train, y_train)
        y_pred_eval = model.predict(X_eval)
        y_pred_train = model.predict(X_train)

    eval_metrics = compute_metrics(y_eval, y_pred_eval)
    train_metrics = compute_metrics(y_train, y_pred_train)
    return model, train_metrics, eval_metrics

In [22]:
def search_with_early_stopping(train_df, eval_df, max_depth, learning_rate,
                                numeric_cols, low_card_cat, high_card_cat,
                                n_estimators_cap=2000, early_stopping_rounds=30):
    """Search-phase fit ONLY (validation month, never the test month): finds
    how many trees a given (max_depth, learning_rate) pair actually wants."""
    X_train, y_train, X_eval, y_eval, numeric_final = prepare_train_eval(
        train_df, eval_df, numeric_cols, high_card_cat
    )

    preprocessor = build_preprocessor(numeric_final, low_card_cat)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
        X_train_t = preprocessor.fit_transform(X_train, y_train)
        X_eval_t = preprocessor.transform(X_eval)

    regressor = XGBRegressor(
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators_cap,   # high cap; early stopping picks the real number
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="rmse",
        early_stopping_rounds=early_stopping_rounds,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    regressor.fit(X_train_t, y_train, eval_set=[(X_eval_t, y_eval)], verbose=False)

    best_iteration = regressor.best_iteration
    y_pred_eval = regressor.predict(X_eval_t, iteration_range=(0, best_iteration + 1))
    y_pred_train = regressor.predict(X_train_t, iteration_range=(0, best_iteration + 1))

    eval_metrics = compute_metrics(y_eval, y_pred_eval)
    train_metrics = compute_metrics(y_train, y_pred_train)
    return best_iteration, train_metrics, eval_metrics

In [23]:
# ---------------------------------------------------------------------------
# Chronological split (Sec. 01)
# ---------------------------------------------------------------------------
months_sorted = sorted(month_period(df["CloseDate"]).unique())
if len(months_sorted) < 3:
    raise ValueError("Need at least 3 distinct months for a train/validation/test split.")

In [24]:
test_month = months_sorted[-1]
val_month = months_sorted[-2]
print(f"Validation month: {val_month}   Test month: {test_month}")

Validation month: 2026-05   Test month: 2026-06


In [25]:
# ---------------------------------------------------------------------------
# LIGHT, evidence-based hyperparameter grid — see docstring above for why
# these specific bounds were chosen. n_estimators is still found via early
# stopping rather than grid-searched.
# ---------------------------------------------------------------------------
PARAM_GRID = {
    "max_depth": [7, 9],
    "learning_rate": [0.03, 0.05, 0.1],
}
N_ESTIMATORS_CAP = 2000
EARLY_STOPPING_ROUNDS = 30

In [26]:
param_combos = [
    dict(zip(PARAM_GRID.keys(), values))
    for values in product(*PARAM_GRID.values())
]
print(f"\nHyperparameter grid: {len(param_combos)} (depth, learning_rate) combinations "
      f"(light, centered on the interior peak found by a prior wide search), "
      f"each capped at {N_ESTIMATORS_CAP} trees with early stopping "
      f"(patience={EARLY_STOPPING_ROUNDS})")


Hyperparameter grid: 6 (depth, learning_rate) combinations (light, centered on the interior peak found by a prior wide search), each capped at 2000 trees with early stopping (patience=30)


In [27]:
def make_regressor(max_depth, learning_rate, n_estimators):
    """Final, non-early-stopped regressor — used only for the last refit
    on the window before the test month, with n_estimators fixed to
    whatever best_iteration the search found."""
    return XGBRegressor(
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=0.8,          # held fixed — not part of the requested tuning scope
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

In [28]:
# ---------------------------------------------------------------------------
# Tune on the validation month only, with the training window fixed at
# FIXED_X months (Sec. 01: never tune on the test month).
# ---------------------------------------------------------------------------
train_window = get_window(df, val_month, FIXED_X)
val_set = get_month(df, val_month)
print(f"\nTuning window: {FIXED_X} months before {val_month} "
      f"({len(train_window)} rows) -> validating on {val_month} ({len(val_set)} rows)")


Tuning window: 24 months before 2026-05 (265957 rows) -> validating on 2026-05 (12019 rows)


In [29]:
search_rows = []
for params in param_combos:
    best_iteration, train_metrics, val_metrics = search_with_early_stopping(
        train_window, val_set,
        params["max_depth"], params["learning_rate"],
        NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT,
        n_estimators_cap=N_ESTIMATORS_CAP,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    gap = train_metrics["R2"] - val_metrics["R2"]
    search_rows.append({
        **params,
        "n_estimators (early-stopped)": best_iteration + 1,
        "Train R2": round(train_metrics["R2"], 4),
        "Val R2": round(val_metrics["R2"], 4),
        "Train-Val gap": round(gap, 4),
    })
    print(f"  depth={params['max_depth']:>2} lr={params['learning_rate']:<4} "
          f"-> stopped at {best_iteration + 1} trees, val R2={val_metrics['R2']:.4f}, "
          f"train-val gap={gap:.4f}")

  depth= 7 lr=0.03 -> stopped at 1347 trees, val R2=0.8645, train-val gap=0.0539
  depth= 7 lr=0.05 -> stopped at 660 trees, val R2=0.8623, train-val gap=0.0494
  depth= 7 lr=0.1  -> stopped at 437 trees, val R2=0.8640, train-val gap=0.0558
  depth= 9 lr=0.03 -> stopped at 900 trees, val R2=0.8672, train-val gap=0.0735
  depth= 9 lr=0.05 -> stopped at 622 trees, val R2=0.8678, train-val gap=0.0767
  depth= 9 lr=0.1  -> stopped at 253 trees, val R2=0.8652, train-val gap=0.0720


In [30]:
search_df = pd.DataFrame(search_rows).sort_values("Val R2", ascending=False).reset_index(drop=True)
print("\n=== Hyperparameter search results (sorted by validation R2) ===\n")
print(search_df.to_string(index=False))


=== Hyperparameter search results (sorted by validation R2) ===

 max_depth  learning_rate  n_estimators (early-stopped)  Train R2  Val R2  Train-Val gap
         9           0.05                           622    0.9445  0.8678         0.0767
         9           0.03                           900    0.9407  0.8672         0.0735
         9           0.10                           253    0.9372  0.8652         0.0720
         7           0.03                          1347    0.9185  0.8645         0.0539
         7           0.10                           437    0.9198  0.8640         0.0558
         7           0.05                           660    0.9117  0.8623         0.0494


In [31]:
best_row = search_df.iloc[0]
best_params = {
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators (early-stopped)"]),
}
print(f"\nSelected hyperparameters: {best_params}")


Selected hyperparameters: {'max_depth': 9, 'learning_rate': 0.05, 'n_estimators': 622}


In [32]:
if best_params["max_depth"] == max(PARAM_GRID["max_depth"]) or \
   best_params["learning_rate"] in (min(PARAM_GRID["learning_rate"]), max(PARAM_GRID["learning_rate"])):
    print("Note: selected value sits at this grid's edge. Given the wider search this grid "
          "was built from, that's expected to be a minor effect rather than a sign the "
          "search needs to be widened again — but worth a quick sanity check if this script "
          "is ever re-run on a materially different data snapshot.")

Note: selected value sits at this grid's edge. Given the wider search this grid was built from, that's expected to be a minor effect rather than a sign the search needs to be widened again — but worth a quick sanity check if this script is ever re-run on a materially different data snapshot.


In [33]:
# ---------------------------------------------------------------------------
# Final fit: retrain with the best hyperparameters on the window immediately
# preceding the test month, then touch the test month exactly once.
# ---------------------------------------------------------------------------
final_train = get_window(df, test_month, FIXED_X)
test_set = get_month(df, test_month)

In [34]:
final_regressor = make_regressor(
    best_params["max_depth"], best_params["learning_rate"], best_params["n_estimators"]
)
final_model, final_train_metrics, final_test_metrics = fit_and_evaluate(
    final_train, test_set, final_regressor, NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT
)

In [35]:
print("\n=== Final Gradient Boosting (XGBoost) — Test Month Result ===")
print(f"Training window: {FIXED_X} months before {test_month} ({len(final_train)} rows)")
print(f"Test month: {test_month} ({len(test_set)} rows)")
print(f"Hyperparameters: {best_params}")
print(f"Train R2: {final_train_metrics['R2']:.4f}")
print(f"Test R2:  {final_test_metrics['R2']:.4f}")
print(f"Train-Test R2 gap: {final_train_metrics['R2'] - final_test_metrics['R2']:.4f}")


=== Final Gradient Boosting (XGBoost) — Test Month Result ===
Training window: 24 months before 2026-06 (264162 rows)
Test month: 2026-06 (12848 rows)
Hyperparameters: {'max_depth': 9, 'learning_rate': 0.05, 'n_estimators': 622}
Train R2: 0.9438
Test R2:  0.8703
Train-Test R2 gap: 0.0735


In [36]:
# ---------------------------------------------------------------------------
# Feature importances — same diagnostic used in Weeks 5-7, to check that
# importance mass sits on intrinsic property features rather than missing-
# value indicators (the pattern that flagged overfitting in the depth=11/
# lr=0.01 run from the wide search).
# ---------------------------------------------------------------------------
feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
importances = final_model.named_steps["regressor"].feature_importances_
top_features = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop 10 features — Gradient Boosting (XGBoost):")
print(top_features.to_string())


Top 10 features — Gradient Boosting (XGBoost):
num__City_target_enc            0.260146
num__LivingArea                 0.101597
num__Bathrooms                  0.089341
cat__PoolPrivateYN_Unknown      0.070804
num__LivingArea_missing         0.065314
cat__Levels_Unknown             0.055993
num__DistrictName_target_enc    0.038284
num__YearBuilt_missing          0.020148
num__PropertyAge_missing        0.018352
cat__FireplaceYN_True           0.017493


In [37]:
"""
Documented model behavior (Gradient Boosting / XGBoost — final, light tuning)
------------------------------------------------------------------------------
  + This grid was deliberately kept small (6 combinations) but is centered
    on a region already shown, by a prior wider search, to contain a real
    interior peak in validation R2 — not on untested boundary values. That
    wider search also showed the highest-R2 combination overall
    (max_depth=11, learning_rate=0.01) won by a margin smaller than
    validation-month noise while nearly doubling the train-test R2 gap and
    shifting feature importance toward missing-value indicators rather than
    intrinsic property features — the signature of overfitting to the
    validation month rather than a genuinely better model. This script
    trades a negligible amount of peak validation R2 for a materially more
    trustworthy (lower-variance, more intrinsically-driven) model.
  + learning_rate controls how much each new tree corrects; early stopping
    lets each rate in this grid use exactly as many trees as it needs.
  - Still more hyperparameter-sensitive than Random Forest in principle;
    the Train-Val gap column in the search table and the final Train-Test
    R2 gap are the diagnostics to watch if this script is ever re-run on a
    new data snapshot and starts selecting a different combination.
"""

'\nDocumented model behavior (Gradient Boosting / XGBoost — final, light tuning)\n------------------------------------------------------------------------------\n  + This grid was deliberately kept small (6 combinations) but is centered\n    on a region already shown, by a prior wider search, to contain a real\n    interior peak in validation R2 — not on untested boundary values. That\n    wider search also showed the highest-R2 combination overall\n    (max_depth=11, learning_rate=0.01) won by a margin smaller than\n    validation-month noise while nearly doubling the train-test R2 gap and\n    shifting feature importance toward missing-value indicators rather than\n    intrinsic property features — the signature of overfitting to the\n    validation month rather than a genuinely better model. This script\n    trades a negligible amount of peak validation R2 for a materially more\n    trustworthy (lower-variance, more intrinsically-driven) model.\n  + learning_rate controls how much e

In [38]:
# ---------------------------------------------------------------------------
# Save hyperparameter search results
# ---------------------------------------------------------------------------
search_df.to_csv("gradient_boosting_search_results.csv", index=False)
print("\nSaved gradient_boosting_search_results.csv")


Saved gradient_boosting_search_results.csv
